In [1]:
!pip install ../input/weights/timm-0.3.1-py3-none-any.whl

Processing /kaggle/input/weights/timm-0.3.1-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/weights/timm-0.3.1-py3-none-any.whl'



In [2]:
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image,  ImageChops
import cv2

from sklearn.metrics import cohen_kappa_score
import timm

device = "cuda:0"


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
# utils
threshold = [0.75, 1.5, 2.5, 3.5]
def regress2class(out):
    prediction = torch.zeros(out.size(0))
    
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu()
    
    return prediction

def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5).cuda()
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze().cpu()
    
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5)).cuda()

    for i in range(out.size(0)):
        if out[i] < 4.:
            l1 = int(math.floor(out[i]))
            l2 = int(math.ceil(out[i]))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.
    
    return pred_prob

In [4]:
# models
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM,self).__init__()
        self.p = Parameter(torch.ones(1)*p)
        self.eps = eps
        self.flatten = flatten
    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x
    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + ', ' + 'eps=' + str(self.eps) + ')'


class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b5_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5
        
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None):
        super(ThreeStage_Model, self).__init__()
        
        self.backbone = timm.models.tf_efficientnet_b4_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
                nn.SiLU(),
                nn.Linear(1000, 500),
                nn.SiLU(),
                nn.Linear(500, 5),
            )
        
        self.regressor = nn.Sequential(
                nn.SiLU(),
                nn.Linear(1000, 500),
                nn.SiLU(),
                nn.Linear(500, 1),
            )

        self.ordinal = nn.Sequential(
                nn.SiLU(),
                nn.Linear(1000, 500),
                nn.SiLU(),
                nn.Linear(500, 4),
            )
        
        self.final_regressor = nn.Sequential(
                nn.SiLU(),
                nn.Linear(10, 1),
            )

    def forward(self, x, final=False):
        x = self.backbone(x)
        
        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)
        
        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out

In [5]:
# image transformation
class photometric_distort(object):
    def __call__(self, image):
        distortions = [FT.adjust_brightness,
                       FT.adjust_contrast,
                       FT.adjust_saturation,
                       FT.adjust_hue]

        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ is 'adjust_hue':
                    adjust_factor = random.uniform(-16 / 255., 16 / 255.)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                # Apply this distortion
                image = d(image, adjust_factor)

        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w
            
        left = (w - new_w)/2
        top = (h - new_h)/2
        right = left + new_w
        bottom = top + new_h

        return image.crop((left, top, right, bottom))

    
class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0,0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
             return image.crop(bbox)

<>:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
/tmp/ipykernel_11/3462085676.py:13: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if d.__name__ is 'adjust_hue':


In [6]:
test_ids = pd.read_csv('../input/aptos2019-blindness-detection/test.csv')
test_ids = np.squeeze(test_ids.values)

input_size = 380

# transform = transforms.Compose([
#                 trim(),
#                 transforms.Resize((input_size, input_size)),
#                 transforms.ToTensor(),
#                 transforms.Normalize(mean=[0.384, 0.258, 0.174],
#                                      std=[0.124, 0.089, 0.094]),
#             ])

transform = transforms.Compose([
                trim(),
                cropTo4_3(),
                transforms.Resize((input_size * 3 // 4, input_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
            ])


net = ThreeStage_Model()
net.load_state_dict(torch.load("../input/weights/B4_3stage_49epoch_CLAHE.pkl"))
net = net.to(device)
# net = torch.load("../input/weights/0.912_tf_efficientnet_b4_ns_regress.pth").to(device)
net.eval()

/tmp/ipykernel_11/2995093931.py:41: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(pretrained=False)


FileNotFoundError: [Errno 2] No such file or directory: '../input/weights/B4_3stage_49epoch_CLAHE.pkl'

In [7]:
submission = []
for i, idx in enumerate(test_ids):
    print(i)
    image_name = "../input/aptos2019-blindness-detection/test_images/{}.png".format(idx)
    img = Image.open(image_name).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)

    _, r_out, _ = net(img)
    pred = regress2class(r_out.data.squeeze(1))
    submission.append([idx, int(pred)])

submission = np.array(submission)

0


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [8]:
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
df.to_csv("submission.csv", index=False)